# N=100, u_max=1280 Adam GPU continuation

This notebook follows `scripts/templates/boilerplate_run.ipynb` unchanged in structure. It operates on the immutable `N100_u1280_adam_gpu_continuation_41k` config: 10 Fourier starts and five perturbations of source run 81268 across the Adam learning-rate, beta1, and beta2 sweep.

**Mandatory agent rule:** use the boilerplate design for run notebooks wherever possible. Do not add, remove, reorder, merge, or redesign sections without the user's explicit permission in the current conversation. Keep cells declarative; reusable behavior belongs in `ofc.notebook_workflow`, never in notebook-local functions or mechanics.

In [ ]:
from ofc.notebook_workflow import RunNotebook

run_name = "N100_u1280_adam_gpu_continuation_41k"
workflow = RunNotebook(run_name)

## Create the immutable config

The config already exists. Leave `Activated=False` to load it; change `run_name` before creating a materially different experiment.

In [ ]:
Activated = False

description = "N=100 u_max=1280 GPU Adam continuation from best run 81268 with fixed source regularization and 41k steps."
reuse_existing = False
parameters = {
    "N": 100,
    "t_interval": 4.0,
    "r_bg": -0.008716,
    "u_isbound": True,
    "v_isbound": True,
    "u_max": 1280.0,
    "v_max": 1000.0,
    "slew_limit": 0.05,
    "optimizer": "adam",
    "schedule": [(1_000, 1.0), (5_000, 0.1), (5_000, 0.1), (30_000, 0.1)],
    "adam_learning_rate": [0.02, 0.05, 0.10],
    "adam_beta1": [0.80, 0.90, 0.95],
    "adam_beta2": [0.99, 0.999],
    "adam_eps": 1e-8,
    "smoothness": 2.5e-7,
    "u_smooth": None,
    "v_smooth": None,
    "sharpness": 2.5e-8,
    "u_sharp": None,
    "v_sharp": None,
    "block_size": 500,
    "J_tol": 1e-5,
    "u_tol": 1e-3,
    "v_tol": 1e-3,
}
runtime = {
    "initialisations": 10,
    "fourier_num_modes": 5,
    "fourier_rms_amplitude": 0.3,
    "fourier_intensity_fraction": 0.3,
    "use_jit": True,
    "use_x64": True,
    "device": "gpu",
    "concurrent_workers": 1,
    "max_cases_per_batch": 18,
    "database": "results/results.sqlite3",
}
initialization_query = {
    "where": {"run_id": 81268, "queue_id": 702019, "status": "complete", "N": 100, "u_max": 1280.0},
    "limit": 1,
    "order_by": "best_score",
    "descending": True,
    "control_kind": "best",
    "resume_optimizer": False,
    "perturbed": True,
    "perturbation_levels": [0.0025, 0.005, 0.01, 0.025, 0.05],
}

config_document = workflow.create_config(
    activated=Activated,
    description=description,
    parameters=parameters,
    runtime=runtime,
    initialization_query=initialization_query,
    reuse_existing=reuse_existing,
)

## Run directly on `bar`'s GPU (detached)

This verifies JAX GPU access and launches a detached process that survives a browser or laptop disconnect.

In [ ]:
Activated = False

queue_id = None
python_executable = None
extra_arguments = []
detached = True
log_path = None

active_queue_id = workflow.run_on_bar_gpu(
    activated=Activated,
    queue_id=queue_id,
    python_executable=python_executable,
    extra_arguments=extra_arguments,
    detached=detached,
    log_path=log_path,
)

## Submit through Slurm (inactive alternative)

In [ ]:
Activated = False

partition = "zen5,epyc"
time = "4-03:00:00"
cpus = 2
memory = "4G"
array = None
array_max_concurrent = None
job_name = None
extra_arguments = []

active_queue_id = workflow.submit_slurm(
    activated=Activated, partition=partition, time=time, cpus=cpus, memory=memory,
    array=array, array_max_concurrent=array_max_concurrent, job_name=job_name,
    extra_arguments=extra_arguments,
)

## Query persisted data

In [ ]:
inherit_config = True
database = None
queue_id = None
config_run_rank = 1
statuses = None
filters = {}
sweep_parameters = ["adam_learning_rate", "adam_beta1", "adam_beta2"]
require_saved_stage = True
limit = None
order_by = "run_id"
descending = False

query_result = workflow.query(
    inherit_config=inherit_config, database=database, queue_id=queue_id,
    config_run_rank=config_run_rank, statuses=statuses, filters=filters,
    sweep_parameters=sweep_parameters, require_saved_stage=require_saved_stage,
    limit=limit, order_by=order_by, descending=descending,
)

## Figure display and saving

In [ ]:
save_figure = None
figure_format = "png"
preview_dpi = 180
save_dpi = 600

## Figure 1 — convergence

In [ ]:
sweep_parameter = "adam_learning_rate"
figure_1 = query_result.plot_convergence(
    sweep_parameter=sweep_parameter, log_base_x=None, log_base_y=None,
    base_x="axis", base_y="axis", x_multiplier=1, y_multiplier=1,
    x_range=None, y_range=None, x_label=None, y_label=None,
)
workflow.present_figure(figure_1, "01_convergence", save_figure=save_figure, figure_format=figure_format)

## Figure 2 — objective strip plot (equally spaced sweep values) and seed sensitivity

In [ ]:
sweep_parameter = "adam_learning_rate"
figure_2 = query_result.plot_distribution(
    sweep_parameter=sweep_parameter, log_base_y=None,
    base_y="axis", y_multiplier=1,
    y_range=None, x_label=None, y_label=None, point_size=24,
    line_alpha=0.22, seed_sensitivity_log_base_y=10,
    seed_sensitivity_base_y=None, seed_sensitivity_y_multiplier=1,
    seed_sensitivity_y_range=None, seed_sensitivity_tolerance=0.01,
)
workflow.present_figure(figure_2, "02_distribution", save_figure=save_figure, figure_format=figure_format)

## Figure 3 — best controls

In [ ]:
sweep_parameter = "adam_learning_rate"
figure_3 = query_result.plot_controls(
    sweep_parameter=sweep_parameter, log_base_x=None, log_base_y=None,
    base_x="axis", base_y="axis", x_multiplier=1, y_multiplier=1,
    x_range=None, y_range=None, x_label=None, y_label=None,
)
workflow.present_figure(figure_3, "03_controls", save_figure=save_figure, figure_format=figure_format)

## Single sweep summary

In [ ]:
single_sweep_parameter = "adam_learning_rate"
history_points = 1200
single_sweep_figure = query_result.plot_single_sweep_summary(
    sweep_parameter=single_sweep_parameter, history_points=history_points,
)
workflow.present_figure(
    single_sweep_figure, "04_single_sweep_summary", save_figure=save_figure,
    figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi,
)

## Double sweep summary

In [ ]:
separate_sweep_parameter = "adam_beta1"
colour_sweep_parameter = "adam_learning_rate"
history_points = 1200
double_sweep_figure = query_result.plot_double_sweep_summary(
    separate_sweep_parameter=separate_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter, history_points=history_points,
)
workflow.present_figure(
    double_sweep_figure, "05_double_sweep_summary", save_figure=save_figure,
    figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi,
)

## Triple sweep summary

In [ ]:
row_sweep_parameter = "adam_beta1"
column_sweep_parameter = "adam_beta2"
colour_sweep_parameter = "adam_learning_rate"
history_points = 1200
triple_sweep_figure = query_result.plot_triple_sweep_summary(
    row_sweep_parameter=row_sweep_parameter, column_sweep_parameter=column_sweep_parameter,
    colour_sweep_parameter=colour_sweep_parameter, history_points=history_points,
)
workflow.present_figure(
    triple_sweep_figure, "06_triple_sweep_summary", save_figure=save_figure,
    figure_format=figure_format, preview_dpi=preview_dpi, save_dpi=save_dpi,
)